# Comparação de Técnicas de Feature Engineering para Modelo de Pricing

Este notebook foi criado para comparar diferentes abordagens de transformação de features no contexto de um modelo de **pricing / funil de simulações que viram propostas**.

## Objetivo

Comparar, em validação temporal e OOT, diferentes formas de transformar variáveis para prever:

```text
target = simulação virou proposta
```

## Técnicas comparadas

1. **WOE + Regressão Logística**
2. **OptBinning + Regressão Logística**
3. **DecisionTreeEncoder + Regressão Logística**
4. **DecisionTreeDiscretiser + WOE + Regressão Logística**
5. **Splines + Regressão Logística**
6. **Polynomial Features + Regressão Logística**
7. **LightGBM baseline**
8. **LightGBM com monotonic constraints**
9. **XGBoost com monotonic constraints**
10. **CatBoost com categóricas nativas**
11. **Target Encoding out-of-fold + modelo de árvore**

## Principais métricas

- ROC AUC
- PR AUC
- KS
- Lift por decil
- Ordenação por grupo de score
- Calibração
- Estabilidade temporal

## Importante

Este notebook foi desenhado para evitar leakage. O correto é:

```text
fit dos transformadores apenas no treino
transform em validação/teste/OOT
```

Nunca calcule WOE, target encoding ou qualquer transformação supervisionada usando target da validação/teste.


In [ ]:
# ============================================================
# 0. Instalação de dependências opcionais
# ============================================================

# Descomente se necessário:
# !pip install feature-engine optbinning lightgbm xgboost catboost category-encoders

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder,
    PolynomialFeatures,
    SplineTransformer
)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    roc_curve
)
from sklearn.model_selection import TimeSeriesSplit, KFold
from sklearn.base import BaseEstimator, TransformerMixin

pd.set_option("display.max_columns", 300)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 240)

print("Setup básico carregado.")


# 1. Configuração do experimento

Ajuste os nomes das colunas abaixo para a sua base.

A estrutura esperada é uma base no nível da **simulação**, com pelo menos:

- identificador do cliente;
- data/hora da simulação;
- target binário indicando se virou proposta;
- variáveis numéricas;
- variáveis categóricas.


In [ ]:
# ============================================================
# 1. Configurações principais
# ============================================================

# Altere conforme sua base
TARGET_COL = "virou_proposta"
DATE_COL = "data_simulacao"
CUSTOMER_COL = "cliente_id"

# Variáveis numéricas candidatas
NUMERIC_VARS = [
    "qtd_simulacoes_ultimos_7d",
    "qtd_simulacoes_ultimos_30d",
    "valor_financiado",
    "valor_entrada",
    "valor_entrada_percentual",
    "prazo",
    "taxa_cliente",
    "ltv",
    "score_cliente",
    "idade_veiculo",
    "tempo_desde_ultima_simulacao"
]

# Variáveis categóricas candidatas
CATEGORICAL_VARS = [
    "canal",
    "tipo_veiculo",
    "produto",
    "uf",
    "perfil_cliente",
    "parceiro",
    "loja"
]

# Variáveis a usar no experimento
# O notebook filtrará automaticamente apenas as colunas existentes no df.
ALL_VARS = NUMERIC_VARS + CATEGORICAL_VARS

# Datas de split temporal
TRAIN_END_DATE = "2026-04-01"
VALID_END_DATE = "2026-05-01"

# Defina se evento positivo é 1
EVENT_VALUE = 1


# 2. Carregamento da base

Substitua a célula abaixo pelo carregamento real da sua base.

Exemplos:

```python
df = pd.read_parquet("../data/processed/base_modelo_pricing.parquet")
```

ou

```python
df = pd.read_csv("../data/processed/base_modelo_pricing.csv")
```


In [ ]:
# ============================================================
# 2. Carregar base
# ============================================================

# Exemplo:
# df = pd.read_parquet("../data/processed/base_modelo_pricing.parquet")

# Placeholder para evitar erro caso você rode sem carregar a base.
# Remova este bloco quando conectar sua base real.
try:
    df
    print("DataFrame df já existe no ambiente.")
except NameError:
    df = pd.DataFrame()
    print("ATENÇÃO: carregue sua base na variável df antes de continuar.")

df.head()


# 3. Funções auxiliares

As funções abaixo calculam:

- AUC;
- PR-AUC;
- KS;
- Brier Score;
- ordenação por grupos de score;
- IV;
- comparação consolidada dos modelos.


In [ ]:
# ============================================================
# 3. Funções auxiliares de avaliação
# ============================================================

def ks_score(y_true, y_score):
    fpr, tpr, _ = roc_curve(y_true, y_score)
    return np.max(tpr - fpr)


def evaluate_predictions(model_name, split_name, y_true, y_score):
    return {
        "model": model_name,
        "split": split_name,
        "roc_auc": roc_auc_score(y_true, y_score),
        "pr_auc": average_precision_score(y_true, y_score),
        "ks": ks_score(y_true, y_score),
        "brier": brier_score_loss(y_true, y_score),
        "n": len(y_true),
        "event_rate": np.mean(y_true)
    }


def evaluate_score_groups(y_true, y_score, n_groups=10):
    temp = pd.DataFrame({
        "target": y_true,
        "score": y_score
    }).copy()

    temp["group"] = pd.qcut(
        temp["score"],
        q=n_groups,
        labels=False,
        duplicates="drop"
    ) + 1

    out = (
        temp
        .groupby("group")
        .agg(
            total=("target", "count"),
            events=("target", "sum"),
            event_rate=("target", "mean"),
            score_min=("score", "min"),
            score_mean=("score", "mean"),
            score_max=("score", "max")
        )
        .reset_index()
    )

    out["event_rate_lift_vs_avg"] = out["event_rate"] / temp["target"].mean()

    return out


def calculate_iv(df, feature_col, target_col, event=1, smoothing=0.5):
    temp = df[[feature_col, target_col]].copy()
    temp[feature_col] = temp[feature_col].fillna("__MISSING__").astype(str)

    grouped = (
        temp
        .groupby(feature_col)[target_col]
        .agg(["count", "sum"])
        .rename(columns={"sum": "events"})
        .reset_index()
    )

    grouped["non_events"] = grouped["count"] - grouped["events"]

    total_events = grouped["events"].sum()
    total_non_events = grouped["non_events"].sum()

    grouped["event_dist"] = (
        grouped["events"] + smoothing
    ) / (total_events + smoothing * len(grouped))

    grouped["non_event_dist"] = (
        grouped["non_events"] + smoothing
    ) / (total_non_events + smoothing * len(grouped))

    grouped["woe"] = np.log(grouped["event_dist"] / grouped["non_event_dist"])
    grouped["iv_component"] = (
        grouped["event_dist"] - grouped["non_event_dist"]
    ) * grouped["woe"]

    grouped["event_rate"] = grouped["events"] / grouped["count"]

    iv = grouped["iv_component"].sum()

    return iv, grouped.sort_values("woe").reset_index(drop=True)


def summarize_iv(df, variables, target_col):
    rows = []
    for var in variables:
        if var in df.columns:
            try:
                iv, _ = calculate_iv(df, var, target_col)
                rows.append({"variable": var, "iv": iv})
            except Exception as e:
                rows.append({"variable": var, "iv": np.nan, "error": str(e)})
    return pd.DataFrame(rows).sort_values("iv", ascending=False)


def safe_existing_columns(df, columns):
    return [c for c in columns if c in df.columns]


# 4. Feature engineering temporal para quantidade de simulações

Esta etapa é opcional, mas útil se você ainda não tiver as features de frequência de simulação.

A lógica correta é criar as contagens usando apenas eventos anteriores à simulação atual.


In [ ]:
# ============================================================
# 4. Criar features de quantidade de simulações por janela
# ============================================================

def add_rolling_simulation_counts(
    df,
    customer_col,
    date_col,
    windows=("1D", "7D", "15D", "30D")
):
    if df.empty:
        return df

    temp = df.copy()
    temp[date_col] = pd.to_datetime(temp[date_col])
    temp = temp.sort_values([customer_col, date_col]).copy()

    for window in windows:
        col_name = f"qtd_simulacoes_ultimos_{window.lower()}"
        counts = (
            temp
            .set_index(date_col)
            .groupby(customer_col)[customer_col]
            .rolling(window)
            .count()
            .reset_index(level=0, drop=True)
        )

        # Remove a simulação atual da contagem
        temp[col_name] = (counts.values - 1).clip(min=0)

    return temp


# Use se precisar criar as features:
# df = add_rolling_simulation_counts(
#     df=df,
#     customer_col=CUSTOMER_COL,
#     date_col=DATE_COL,
#     windows=("1D", "7D", "15D", "30D")
# )

# df.head()


# 5. Split temporal

O split temporal é fundamental para medir se as transformações supervisionadas são estáveis fora do tempo.

A recomendação é:

```text
treino     = meses antigos
validação  = mês seguinte
teste/OOT  = meses mais recentes
```


In [ ]:
# ============================================================
# 5. Split temporal
# ============================================================

if df.empty:
    print("Carregue a base em df antes de continuar.")
else:
    df = df.copy()
    df[DATE_COL] = pd.to_datetime(df[DATE_COL])
    df = df.sort_values(DATE_COL).copy()

    # Filtra variáveis existentes
    numeric_vars = safe_existing_columns(df, NUMERIC_VARS)
    categorical_vars = safe_existing_columns(df, CATEGORICAL_VARS)
    all_vars = numeric_vars + categorical_vars

    print("Numéricas existentes:", numeric_vars)
    print("Categóricas existentes:", categorical_vars)

    train = df[df[DATE_COL] < TRAIN_END_DATE].copy()

    valid = df[
        (df[DATE_COL] >= TRAIN_END_DATE) &
        (df[DATE_COL] < VALID_END_DATE)
    ].copy()

    test = df[df[DATE_COL] >= VALID_END_DATE].copy()

    X_train = train[all_vars].copy()
    y_train = train[TARGET_COL].astype(int).copy()

    X_valid = valid[all_vars].copy()
    y_valid = valid[TARGET_COL].astype(int).copy()

    X_test = test[all_vars].copy()
    y_test = test[TARGET_COL].astype(int).copy()

    print("Train:", train.shape, "Event rate:", y_train.mean())
    print("Valid:", valid.shape, "Event rate:", y_valid.mean())
    print("Test:", test.shape, "Event rate:", y_test.mean())


# 6. Baseline 1 — WOE + Regressão Logística

Este é o benchmark interpretável mais próximo de um scorecard.

Fluxo:

```text
numéricas → EqualFrequencyDiscretiser
categóricas → mantidas
todas → WoEEncoder
modelo → LogisticRegression
```


In [ ]:
# ============================================================
# 6. WOE + Logistic Regression
# ============================================================

results = []
score_tables = {}
models = {}

try:
    from feature_engine.discretisation import EqualFrequencyDiscretiser
    from feature_engine.encoding import WoEEncoder

    pipe_woe = Pipeline(steps=[
        (
            "discretizer",
            EqualFrequencyDiscretiser(
                q=5,
                variables=numeric_vars,
                return_object=True
            )
        ),
        (
            "woe_encoder",
            WoEEncoder(
                variables=all_vars,
                unseen="ignore"
            )
        ),
        (
            "model",
            LogisticRegression(
                penalty="l2",
                C=1.0,
                solver="liblinear",
                max_iter=1000
            )
        )
    ])

    pipe_woe.fit(X_train, y_train)

    valid_pred_woe = pipe_woe.predict_proba(X_valid)[:, 1]
    test_pred_woe = pipe_woe.predict_proba(X_test)[:, 1]

    results.append(evaluate_predictions("WOE + LogReg", "valid", y_valid, valid_pred_woe))
    results.append(evaluate_predictions("WOE + LogReg", "test", y_test, test_pred_woe))

    score_tables["WOE + LogReg"] = evaluate_score_groups(y_test, test_pred_woe)
    models["WOE + LogReg"] = pipe_woe

    print("WOE + LogReg treinado com sucesso.")

except Exception as e:
    print("Erro no WOE + LogReg:", e)


# 7. Baseline 2 — OptBinning + Regressão Logística

O `OptBinning` cria bins supervisionados de forma mais robusta e pode transformar para WOE.

Esta abordagem é interessante quando você quer algo mais próximo de um scorecard profissional.


In [ ]:
# ============================================================
# 7. OptBinning + Logistic Regression
# ============================================================

try:
    from optbinning import BinningProcess

    # O BinningProcess cuida das variáveis numéricas e categóricas.
    binning_process = BinningProcess(
        variable_names=all_vars,
        categorical_variables=categorical_vars
    )

    pipe_optbinning = Pipeline(steps=[
        ("binning", binning_process),
        (
            "model",
            LogisticRegression(
                penalty="l2",
                C=1.0,
                solver="liblinear",
                max_iter=1000
            )
        )
    ])

    pipe_optbinning.fit(X_train, y_train)

    valid_pred_opt = pipe_optbinning.predict_proba(X_valid)[:, 1]
    test_pred_opt = pipe_optbinning.predict_proba(X_test)[:, 1]

    results.append(evaluate_predictions("OptBinning + LogReg", "valid", y_valid, valid_pred_opt))
    results.append(evaluate_predictions("OptBinning + LogReg", "test", y_test, test_pred_opt))

    score_tables["OptBinning + LogReg"] = evaluate_score_groups(y_test, test_pred_opt)
    models["OptBinning + LogReg"] = pipe_optbinning

    print("OptBinning + LogReg treinado com sucesso.")

except Exception as e:
    print("OptBinning não executado. Detalhe:", e)


# 8. Baseline 3 — DecisionTreeEncoder + Regressão Logística

O `DecisionTreeEncoder` transforma cada variável usando a predição de uma árvore treinada com aquela variável para prever o target.

É uma alternativa supervisionada ao WOE.


In [ ]:
# ============================================================
# 8. DecisionTreeEncoder + Logistic Regression
# ============================================================

try:
    from feature_engine.encoding import DecisionTreeEncoder

    pipe_tree_encoder = Pipeline(steps=[
        (
            "tree_encoder",
            DecisionTreeEncoder(
                variables=all_vars,
                regression=False,
                cv=3,
                scoring="roc_auc",
                ignore_format=True,
                random_state=42,
                unseen="ignore",
                param_grid={
                    "max_depth": [1, 2, 3, 4],
                    "min_samples_leaf": [100, 500, 1000]
                }
            )
        ),
        (
            "model",
            LogisticRegression(
                penalty="l2",
                C=1.0,
                solver="liblinear",
                max_iter=1000
            )
        )
    ])

    pipe_tree_encoder.fit(X_train, y_train)

    valid_pred_dte = pipe_tree_encoder.predict_proba(X_valid)[:, 1]
    test_pred_dte = pipe_tree_encoder.predict_proba(X_test)[:, 1]

    results.append(evaluate_predictions("DecisionTreeEncoder + LogReg", "valid", y_valid, valid_pred_dte))
    results.append(evaluate_predictions("DecisionTreeEncoder + LogReg", "test", y_test, test_pred_dte))

    score_tables["DecisionTreeEncoder + LogReg"] = evaluate_score_groups(y_test, test_pred_dte)
    models["DecisionTreeEncoder + LogReg"] = pipe_tree_encoder

    print("DecisionTreeEncoder + LogReg treinado com sucesso.")

except Exception as e:
    print("DecisionTreeEncoder não executado. Detalhe:", e)


# 9. Baseline 4 — DecisionTreeDiscretiser + WOE + Regressão Logística

Esta abordagem costuma ser muito interessante para scorecards:

```text
árvore cria os buckets supervisionados
WOE transforma buckets em evidência
regressão logística combina os sinais
```


In [ ]:
# ============================================================
# 9. DecisionTreeDiscretiser + WOE + Logistic Regression
# ============================================================

try:
    from feature_engine.discretisation import DecisionTreeDiscretiser
    from feature_engine.encoding import WoEEncoder

    pipe_tree_binning_woe = Pipeline(steps=[
        (
            "tree_discretizer",
            DecisionTreeDiscretiser(
                variables=numeric_vars,
                regression=False,
                cv=3,
                scoring="roc_auc",
                param_grid={
                    "max_depth": [1, 2, 3],
                    "min_samples_leaf": [100, 500, 1000]
                },
                random_state=42
            )
        ),
        (
            "woe_encoder",
            WoEEncoder(
                variables=all_vars,
                unseen="ignore"
            )
        ),
        (
            "model",
            LogisticRegression(
                penalty="l2",
                C=1.0,
                solver="liblinear",
                max_iter=1000
            )
        )
    ])

    pipe_tree_binning_woe.fit(X_train, y_train)

    valid_pred_tbw = pipe_tree_binning_woe.predict_proba(X_valid)[:, 1]
    test_pred_tbw = pipe_tree_binning_woe.predict_proba(X_test)[:, 1]

    results.append(evaluate_predictions("TreeBinning + WOE + LogReg", "valid", y_valid, valid_pred_tbw))
    results.append(evaluate_predictions("TreeBinning + WOE + LogReg", "test", y_test, test_pred_tbw))

    score_tables["TreeBinning + WOE + LogReg"] = evaluate_score_groups(y_test, test_pred_tbw)
    models["TreeBinning + WOE + LogReg"] = pipe_tree_binning_woe

    print("TreeBinning + WOE + LogReg treinado com sucesso.")

except Exception as e:
    print("TreeBinning + WOE não executado. Detalhe:", e)


# 10. Baseline 5 — Splines + Regressão Logística

Splines são úteis para variáveis contínuas importantes, como:

- taxa;
- prazo;
- LTV;
- entrada percentual;
- score;
- idade do veículo.

Elas capturam relações não lineares suaves.


In [ ]:
# ============================================================
# 10. Splines + Logistic Regression
# ============================================================

try:
    spline_vars = [
        c for c in [
            "taxa_cliente",
            "valor_entrada_percentual",
            "ltv",
            "prazo",
            "score_cliente",
            "idade_veiculo",
            "qtd_simulacoes_ultimos_7d",
            "qtd_simulacoes_ultimos_30d"
        ]
        if c in numeric_vars
    ]

    other_numeric = [c for c in numeric_vars if c not in spline_vars]

    spline_preprocessor = ColumnTransformer(
        transformers=[
            (
                "spline",
                Pipeline(steps=[
                    ("scaler", StandardScaler()),
                    ("spline", SplineTransformer(
                        degree=3,
                        n_knots=5,
                        include_bias=False
                    )),
                    ("scaler_after", StandardScaler())
                ]),
                spline_vars
            ),
            (
                "num",
                Pipeline(steps=[
                    ("scaler", StandardScaler())
                ]),
                other_numeric
            ),
            (
                "cat",
                OneHotEncoder(
                    handle_unknown="ignore",
                    min_frequency=0.01
                ),
                categorical_vars
            )
        ],
        remainder="drop"
    )

    pipe_splines = Pipeline(steps=[
        ("preprocessor", spline_preprocessor),
        (
            "model",
            LogisticRegression(
                penalty="l2",
                C=0.5,
                solver="lbfgs",
                max_iter=3000
            )
        )
    ])

    pipe_splines.fit(X_train, y_train)

    valid_pred_spline = pipe_splines.predict_proba(X_valid)[:, 1]
    test_pred_spline = pipe_splines.predict_proba(X_test)[:, 1]

    results.append(evaluate_predictions("Splines + LogReg", "valid", y_valid, valid_pred_spline))
    results.append(evaluate_predictions("Splines + LogReg", "test", y_test, test_pred_spline))

    score_tables["Splines + LogReg"] = evaluate_score_groups(y_test, test_pred_spline)
    models["Splines + LogReg"] = pipe_splines

    print("Splines + LogReg treinado com sucesso.")

except Exception as e:
    print("Splines + LogReg não executado. Detalhe:", e)


# 11. Baseline 6 — Polynomial Features + Regressão Logística

Polynomial features podem capturar efeitos em U ou relações curvas simples.

Recomendação:

- começar com `degree=2`;
- usar poucas variáveis;
- sempre usar regularização.


In [ ]:
# ============================================================
# 11. Polynomial Features + Logistic Regression
# ============================================================

try:
    poly_vars = [
        c for c in [
            "taxa_cliente",
            "valor_entrada_percentual",
            "ltv",
            "prazo",
            "qtd_simulacoes_ultimos_7d",
            "qtd_simulacoes_ultimos_30d",
            "tempo_desde_ultima_simulacao"
        ]
        if c in numeric_vars
    ]

    other_numeric = [c for c in numeric_vars if c not in poly_vars]

    poly_preprocessor = ColumnTransformer(
        transformers=[
            (
                "poly",
                Pipeline(steps=[
                    ("scaler", StandardScaler()),
                    ("poly", PolynomialFeatures(
                        degree=2,
                        include_bias=False,
                        interaction_only=False
                    )),
                    ("scaler_after", StandardScaler())
                ]),
                poly_vars
            ),
            (
                "num",
                Pipeline(steps=[
                    ("scaler", StandardScaler())
                ]),
                other_numeric
            ),
            (
                "cat",
                OneHotEncoder(
                    handle_unknown="ignore",
                    min_frequency=0.01
                ),
                categorical_vars
            )
        ],
        remainder="drop"
    )

    pipe_poly = Pipeline(steps=[
        ("preprocessor", poly_preprocessor),
        (
            "model",
            LogisticRegression(
                penalty="l2",
                C=0.5,
                solver="lbfgs",
                max_iter=3000
            )
        )
    ])

    pipe_poly.fit(X_train, y_train)

    valid_pred_poly = pipe_poly.predict_proba(X_valid)[:, 1]
    test_pred_poly = pipe_poly.predict_proba(X_test)[:, 1]

    results.append(evaluate_predictions("Polynomial + LogReg", "valid", y_valid, valid_pred_poly))
    results.append(evaluate_predictions("Polynomial + LogReg", "test", y_test, test_pred_poly))

    score_tables["Polynomial + LogReg"] = evaluate_score_groups(y_test, test_pred_poly)
    models["Polynomial + LogReg"] = pipe_poly

    print("Polynomial + LogReg treinado com sucesso.")

except Exception as e:
    print("Polynomial + LogReg não executado. Detalhe:", e)


# 12. Encoder opcional — Target Encoding Out-of-Fold

Target encoding é muito útil para modelos baseados em árvores, principalmente em variáveis categóricas de alta cardinalidade.

Atenção: target encoding precisa ser feito de forma out-of-fold no treino para evitar leakage.


In [ ]:
# ============================================================
# 12. Target Encoding Out-of-Fold simples
# ============================================================

class SimpleOOFTargetEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, variables, n_splits=5, smoothing=20, random_state=42):
        self.variables = variables
        self.n_splits = n_splits
        self.smoothing = smoothing
        self.random_state = random_state
        self.global_mean_ = None
        self.maps_ = {}

    def fit(self, X, y):
        X = X.copy()
        y = pd.Series(y).reset_index(drop=True)
        X = X.reset_index(drop=True)

        self.global_mean_ = y.mean()
        self.maps_ = {}

        for var in self.variables:
            temp = pd.DataFrame({
                var: X[var].fillna("__MISSING__").astype(str),
                "target": y
            })

            stats = temp.groupby(var)["target"].agg(["mean", "count"])
            smooth = (
                (stats["mean"] * stats["count"] + self.global_mean_ * self.smoothing)
                / (stats["count"] + self.smoothing)
            )

            self.maps_[var] = smooth.to_dict()

        return self

    def transform(self, X):
        X = X.copy()

        for var in self.variables:
            encoded = (
                X[var]
                .fillna("__MISSING__")
                .astype(str)
                .map(self.maps_.get(var, {}))
                .fillna(self.global_mean_)
            )
            X[var] = encoded

        return X

    def fit_transform_oof(self, X, y):
        X = X.copy().reset_index(drop=True)
        y = pd.Series(y).reset_index(drop=True)

        X_encoded = X.copy()
        self.global_mean_ = y.mean()

        kf = KFold(
            n_splits=self.n_splits,
            shuffle=True,
            random_state=self.random_state
        )

        for var in self.variables:
            X_encoded[var] = np.nan

        for train_idx, val_idx in kf.split(X):
            X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
            y_tr = y.iloc[train_idx]

            for var in self.variables:
                temp = pd.DataFrame({
                    var: X_tr[var].fillna("__MISSING__").astype(str),
                    "target": y_tr
                })

                stats = temp.groupby(var)["target"].agg(["mean", "count"])
                smooth = (
                    (stats["mean"] * stats["count"] + self.global_mean_ * self.smoothing)
                    / (stats["count"] + self.smoothing)
                )

                X_encoded.loc[val_idx, var] = (
                    X_val[var]
                    .fillna("__MISSING__")
                    .astype(str)
                    .map(smooth.to_dict())
                    .fillna(self.global_mean_)
                )

        # Fit final maps using full training data
        self.fit(X, y)

        return X_encoded


# 13. Baseline 7 — LightGBM com Target Encoding OOF

Modelo de árvore forte para benchmark de performance.

Se sua base tiver muitas categóricas de alta cardinalidade, o target encoding OOF pode ajudar.


In [ ]:
# ============================================================
# 13. LightGBM + Target Encoding OOF
# ============================================================

try:
    from lightgbm import LGBMClassifier

    te = SimpleOOFTargetEncoder(
        variables=categorical_vars,
        n_splits=5,
        smoothing=30,
        random_state=42
    )

    X_train_te = X_train.copy()
    X_valid_te = X_valid.copy()
    X_test_te = X_test.copy()

    if categorical_vars:
        X_train_te = te.fit_transform_oof(X_train_te, y_train)
        X_valid_te = te.transform(X_valid_te)
        X_test_te = te.transform(X_test_te)

    lgbm_te = LGBMClassifier(
        objective="binary",
        n_estimators=500,
        learning_rate=0.03,
        max_depth=-1,
        num_leaves=31,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42
    )

    lgbm_te.fit(X_train_te, y_train)

    valid_pred_lgbm_te = lgbm_te.predict_proba(X_valid_te)[:, 1]
    test_pred_lgbm_te = lgbm_te.predict_proba(X_test_te)[:, 1]

    results.append(evaluate_predictions("LightGBM + OOF TargetEnc", "valid", y_valid, valid_pred_lgbm_te))
    results.append(evaluate_predictions("LightGBM + OOF TargetEnc", "test", y_test, test_pred_lgbm_te))

    score_tables["LightGBM + OOF TargetEnc"] = evaluate_score_groups(y_test, test_pred_lgbm_te)
    models["LightGBM + OOF TargetEnc"] = lgbm_te

    print("LightGBM + OOF TargetEnc treinado com sucesso.")

except Exception as e:
    print("LightGBM + Target Encoding não executado. Detalhe:", e)


# 14. Monotonic Constraints

Para modelos baseados em árvores, a ordenação por score pode ser boa, mas a monotonicidade por variável não é garantida.

Se você quer impor regras como:

```text
taxa_cliente aumenta → probabilidade de virar proposta não deve aumentar
score_cliente aumenta → probabilidade de virar proposta não deve diminuir
```

use monotonic constraints.

Abaixo, ajuste o dicionário conforme sua regra de negócio.


In [ ]:
# ============================================================
# 14. Configuração de monotonic constraints
# ============================================================

# Convenção:
#  1  = monotonicamente crescente
# -1  = monotonicamente decrescente
#  0  = sem restrição

MONOTONIC_RULES = {
    "taxa_cliente": -1,
    "valor_parcela": -1,
    "tempo_desde_ultima_simulacao": -1,
    "score_cliente": 1,
    "valor_entrada_percentual": 1,

    # Cuidado: essas podem ser não monotônicas.
    "qtd_simulacoes_ultimos_7d": 0,
    "qtd_simulacoes_ultimos_30d": 0,
    "prazo": 0,
    "ltv": 0,
    "idade_veiculo": 0
}

def build_monotonic_list(feature_names, rules):
    return [rules.get(f, 0) for f in feature_names]

monotonic_constraints = build_monotonic_list(all_vars, MONOTONIC_RULES)

list(zip(all_vars, monotonic_constraints))


# 15. Baseline 8 — LightGBM com Monotonic Constraints

Use este modelo quando quiser equilibrar:

- performance;
- ordenação;
- coerência de negócio;
- controle monotônico por variável.


In [ ]:
# ============================================================
# 15. LightGBM com Monotonic Constraints
# ============================================================

try:
    from lightgbm import LGBMClassifier

    # Para LightGBM, vamos usar target encoding nas categóricas
    te_mono = SimpleOOFTargetEncoder(
        variables=categorical_vars,
        n_splits=5,
        smoothing=30,
        random_state=42
    )

    X_train_mono = X_train.copy()
    X_valid_mono = X_valid.copy()
    X_test_mono = X_test.copy()

    if categorical_vars:
        X_train_mono = te_mono.fit_transform_oof(X_train_mono, y_train)
        X_valid_mono = te_mono.transform(X_valid_mono)
        X_test_mono = te_mono.transform(X_test_mono)

    monotonic_constraints = build_monotonic_list(all_vars, MONOTONIC_RULES)

    lgbm_mono = LGBMClassifier(
        objective="binary",
        n_estimators=500,
        learning_rate=0.03,
        num_leaves=16,
        max_depth=4,
        subsample=0.8,
        colsample_bytree=0.8,
        monotone_constraints=monotonic_constraints,
        random_state=42
    )

    lgbm_mono.fit(X_train_mono, y_train)

    valid_pred_lgbm_mono = lgbm_mono.predict_proba(X_valid_mono)[:, 1]
    test_pred_lgbm_mono = lgbm_mono.predict_proba(X_test_mono)[:, 1]

    results.append(evaluate_predictions("LightGBM Monotonic", "valid", y_valid, valid_pred_lgbm_mono))
    results.append(evaluate_predictions("LightGBM Monotonic", "test", y_test, test_pred_lgbm_mono))

    score_tables["LightGBM Monotonic"] = evaluate_score_groups(y_test, test_pred_lgbm_mono)
    models["LightGBM Monotonic"] = lgbm_mono

    print("LightGBM Monotonic treinado com sucesso.")

except Exception as e:
    print("LightGBM Monotonic não executado. Detalhe:", e)


# 16. Baseline 9 — XGBoost com Monotonic Constraints

Alternativa forte ao LightGBM.

Atenção: para XGBoost também vamos usar categóricas previamente tratadas por target encoding OOF.


In [ ]:
# ============================================================
# 16. XGBoost com Monotonic Constraints
# ============================================================

try:
    from xgboost import XGBClassifier

    te_xgb = SimpleOOFTargetEncoder(
        variables=categorical_vars,
        n_splits=5,
        smoothing=30,
        random_state=42
    )

    X_train_xgb = X_train.copy()
    X_valid_xgb = X_valid.copy()
    X_test_xgb = X_test.copy()

    if categorical_vars:
        X_train_xgb = te_xgb.fit_transform_oof(X_train_xgb, y_train)
        X_valid_xgb = te_xgb.transform(X_valid_xgb)
        X_test_xgb = te_xgb.transform(X_test_xgb)

    constraints_tuple = tuple(build_monotonic_list(all_vars, MONOTONIC_RULES))

    xgb_mono = XGBClassifier(
        objective="binary:logistic",
        eval_metric="auc",
        n_estimators=500,
        learning_rate=0.03,
        max_depth=4,
        subsample=0.8,
        colsample_bytree=0.8,
        monotone_constraints=constraints_tuple,
        random_state=42,
        tree_method="hist"
    )

    xgb_mono.fit(X_train_xgb, y_train)

    valid_pred_xgb_mono = xgb_mono.predict_proba(X_valid_xgb)[:, 1]
    test_pred_xgb_mono = xgb_mono.predict_proba(X_test_xgb)[:, 1]

    results.append(evaluate_predictions("XGBoost Monotonic", "valid", y_valid, valid_pred_xgb_mono))
    results.append(evaluate_predictions("XGBoost Monotonic", "test", y_test, test_pred_xgb_mono))

    score_tables["XGBoost Monotonic"] = evaluate_score_groups(y_test, test_pred_xgb_mono)
    models["XGBoost Monotonic"] = xgb_mono

    print("XGBoost Monotonic treinado com sucesso.")

except Exception as e:
    print("XGBoost Monotonic não executado. Detalhe:", e)


# 17. Baseline 10 — CatBoost com categóricas nativas

CatBoost costuma ser forte quando há muitas variáveis categóricas.

Ele pode consumir variáveis categóricas diretamente.


In [ ]:
# ============================================================
# 17. CatBoost com categóricas nativas
# ============================================================

try:
    from catboost import CatBoostClassifier, Pool

    X_train_cat = X_train.copy()
    X_valid_cat = X_valid.copy()
    X_test_cat = X_test.copy()

    # CatBoost precisa dos índices das categóricas dentro de X
    cat_features_idx = [
        X_train_cat.columns.get_loc(c)
        for c in categorical_vars
        if c in X_train_cat.columns
    ]

    # Preenche nulos
    for c in categorical_vars:
        if c in X_train_cat.columns:
            X_train_cat[c] = X_train_cat[c].fillna("__MISSING__").astype(str)
            X_valid_cat[c] = X_valid_cat[c].fillna("__MISSING__").astype(str)
            X_test_cat[c] = X_test_cat[c].fillna("__MISSING__").astype(str)

    for c in numeric_vars:
        if c in X_train_cat.columns:
            median_value = X_train_cat[c].median()
            X_train_cat[c] = X_train_cat[c].fillna(median_value)
            X_valid_cat[c] = X_valid_cat[c].fillna(median_value)
            X_test_cat[c] = X_test_cat[c].fillna(median_value)

    cat_model = CatBoostClassifier(
        loss_function="Logloss",
        eval_metric="AUC",
        iterations=500,
        learning_rate=0.03,
        depth=4,
        random_seed=42,
        verbose=False
    )

    cat_model.fit(
        X_train_cat,
        y_train,
        cat_features=cat_features_idx,
        eval_set=(X_valid_cat, y_valid),
        use_best_model=True
    )

    valid_pred_cat = cat_model.predict_proba(X_valid_cat)[:, 1]
    test_pred_cat = cat_model.predict_proba(X_test_cat)[:, 1]

    results.append(evaluate_predictions("CatBoost Native Cat", "valid", y_valid, valid_pred_cat))
    results.append(evaluate_predictions("CatBoost Native Cat", "test", y_test, test_pred_cat))

    score_tables["CatBoost Native Cat"] = evaluate_score_groups(y_test, test_pred_cat)
    models["CatBoost Native Cat"] = cat_model

    print("CatBoost treinado com sucesso.")

except Exception as e:
    print("CatBoost não executado. Detalhe:", e)


# 18. Comparação consolidada dos modelos

A tabela abaixo consolida as principais métricas.

Recomendações de leitura:

- **PR-AUC** é muito importante se o target for raro.
- **KS** ajuda a avaliar separação entre eventos e não-eventos.
- **Brier** avalia calibração; menor é melhor.
- Compare sempre validação e teste/OOT.


In [ ]:
# ============================================================
# 18. Resultado consolidado
# ============================================================

results_df = pd.DataFrame(results)

if not results_df.empty:
    display(
        results_df
        .sort_values(["split", "pr_auc"], ascending=[True, False])
        .reset_index(drop=True)
    )
else:
    print("Nenhum modelo foi executado ainda.")


# 19. Ordenação por grupo de score

Aqui você avalia se o modelo está ordenando corretamente os grupos de propensão.

A taxa de evento deve crescer conforme o grupo aumenta.


In [ ]:
# ============================================================
# 19. Ver ordenação por modelo
# ============================================================

for model_name, table in score_tables.items():
    print("\n" + "=" * 100)
    print(model_name)
    display(table)


# 20. Comparação visual da ordenação

Este gráfico compara a taxa de evento por decil dos modelos no teste/OOT.


In [ ]:
# ============================================================
# 20. Plot da ordenação por decil
# ============================================================

import matplotlib.pyplot as plt

if score_tables:
    comparison_rows = []

    for model_name, table in score_tables.items():
        temp = table.copy()
        temp["model"] = model_name
        comparison_rows.append(temp)

    score_comparison = pd.concat(comparison_rows, axis=0)

    plt.figure(figsize=(12, 6))

    for model_name in score_comparison["model"].unique():
        temp = score_comparison[score_comparison["model"] == model_name]
        plt.plot(temp["group"], temp["event_rate"], marker="o", label=model_name)

    plt.xlabel("Grupo de score")
    plt.ylabel("Taxa de evento")
    plt.title("Ordenação por grupo de score no teste/OOT")
    plt.legend()
    plt.grid(True)
    plt.show()
else:
    print("Nenhuma tabela de score disponível.")


# 21. Calibração por grupo

Além de ordenar, o modelo precisa estar calibrado.

Aqui comparamos:

```text
score médio previsto vs taxa real observada
```


In [ ]:
# ============================================================
# 21. Calibração por grupo
# ============================================================

if score_tables:
    for model_name, table in score_tables.items():
        print("\n" + "=" * 100)
        print(f"Calibração por grupo — {model_name}")

        calib = table[[
            "group",
            "score_mean",
            "event_rate",
            "total"
        ]].copy()

        calib["calibration_error"] = calib["score_mean"] - calib["event_rate"]

        display(calib)
else:
    print("Nenhuma tabela de score disponível.")


# 22. IV das variáveis transformadas

Esta seção é útil principalmente para:

- WOE;
- DecisionTreeEncoder;
- TreeBinning + WOE;
- Target Encoding.

O objetivo é medir o poder individual de separação das variáveis transformadas.


In [ ]:
# ============================================================
# 22. IV das transformações supervisionadas
# ============================================================

def transformed_feature_iv(model_name, pipeline, X, y, variables, target_col=TARGET_COL):
    try:
        # Remove o último step se for modelo
        if isinstance(pipeline, Pipeline) and "model" in pipeline.named_steps:
            transformer_steps = pipeline.steps[:-1]
            transformer = Pipeline(transformer_steps)
            Xt = transformer.transform(X)
        else:
            Xt = pipeline.transform(X)

        if not isinstance(Xt, pd.DataFrame):
            # Alguns transformers retornam numpy array; nesse caso não temos nomes diretamente.
            print(f"{model_name}: transformação retornou array. IV por variável não calculado automaticamente.")
            return pd.DataFrame()

        temp = Xt.copy()
        temp[target_col] = np.array(y)

        return summarize_iv(temp, [c for c in temp.columns if c != target_col], target_col)

    except Exception as e:
        print(f"Erro ao calcular IV para {model_name}: {e}")
        return pd.DataFrame()


iv_outputs = {}

for model_name in [
    "WOE + LogReg",
    "DecisionTreeEncoder + LogReg",
    "TreeBinning + WOE + LogReg"
]:
    if model_name in models:
        print("\n" + "=" * 100)
        print(model_name)
        iv_table = transformed_feature_iv(
            model_name=model_name,
            pipeline=models[model_name],
            X=X_test,
            y=y_test,
            variables=all_vars
        )
        iv_outputs[model_name] = iv_table
        display(iv_table.head(30))


# 23. Estabilidade temporal mês a mês

Esta análise ajuda a identificar se a ordenação do modelo é estável ao longo dos meses.

Você pode aplicá-la ao modelo vencedor.


In [ ]:
# ============================================================
# 23. Estabilidade temporal mês a mês
# ============================================================

def monthly_stability(df_base, date_col, target_col, score_col, n_groups=10):
    temp = df_base[[date_col, target_col, score_col]].copy()
    temp[date_col] = pd.to_datetime(temp[date_col])
    temp["month"] = temp[date_col].dt.to_period("M").astype(str)

    rows = []

    for month, df_m in temp.groupby("month"):
        if df_m[target_col].nunique() < 2 or len(df_m) < n_groups:
            continue

        group_table = evaluate_score_groups(
            y_true=df_m[target_col],
            y_score=df_m[score_col],
            n_groups=n_groups
        )

        group_table["month"] = month
        rows.append(group_table)

    if rows:
        return pd.concat(rows, axis=0).reset_index(drop=True)

    return pd.DataFrame()


# Exemplo de uso com um modelo específico:
# test_with_score = test.copy()
# test_with_score["score"] = test_pred_woe
# stability = monthly_stability(test_with_score, DATE_COL, TARGET_COL, "score")
# display(stability)


# 24. Como escolher o vencedor

Sugestão de critério:

1. Modelo com boa performance em teste/OOT.
2. Boa ordenação por grupo de score.
3. Boa calibração.
4. Estabilidade temporal.
5. Coerência com regras de negócio.
6. Simplicidade operacional.

## Recomendações práticas

### Se o foco for explicabilidade

Priorize:

```text
WOE + LogReg
OptBinning + LogReg
TreeBinning + WOE + LogReg
```

### Se o foco for performance

Priorize:

```text
CatBoost
LightGBM + Target Encoding OOF
XGBoost
```

### Se o foco for performance com controle de negócio

Priorize:

```text
LightGBM/XGBoost/CatBoost com monotonic constraints
```

### Se o foco for capturar não linearidade mantendo modelo linear

Priorize:

```text
Splines + LogReg
Polynomial degree 2 + LogReg
```


# 25. Checklist final contra leakage

Antes de confiar nos resultados:

- [ ] WOE foi ajustado somente no treino?
- [ ] Target encoding foi out-of-fold no treino?
- [ ] Features de simulação usam apenas informações anteriores à simulação atual?
- [ ] Split é temporal?
- [ ] OOT representa período mais recente?
- [ ] Variáveis pós-evento foram removidas?
- [ ] Política comercial/campanha futura não vazou para o passado?
- [ ] O modelo mantém ordenação por score no OOT?
- [ ] O modelo está calibrado?
- [ ] A monotonicidade imposta faz sentido de negócio?
